In [ ]:
# Project Review Status Classification - Data Preparation
# ======================================================

# This notebook handles the data preparation for the project review status classification model
# It includes text preprocessing, feature extraction from JSON fields, date feature engineering,
# and handling missing values.

# Table of Contents:
# 1. Import Libraries and Load Data
# 2. Exploratory Data Analysis
# 3. Text Preprocessing
# 4. JSON Field Processing
# 5. Date Feature Engineering
# 6. Handle Missing Values
# 7. Feature Consolidation
# 8. Export Processed Dataset

# 1. Import Libraries and Load Data
# ---------------------------------

import pandas as pd
import numpy as np
import json
import re
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# NLP libraries
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import string
import textstat  # For readability metrics

# Download necessary NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('vader_lexicon')

# Set the display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Load the dataset
# Replace 'project_data.csv' with your actual file path
df = pd.read_csv("../synthetic_projects_data.csv")

# Display basic information
print(f"Dataset Shape: {df.shape}")
df.info()

: 

In [ ]:
# 2. Exploratory Data Analysis
# ----------------------------

# Display class distribution
print("\nReview Status Distribution:")
status_counts = df['review_status'].value_counts()
print(status_counts)

# Plot the distribution
plt.figure(figsize=(10, 6))
sns.countplot(x='review_status', data=df)
plt.title('Distribution of Review Status')
plt.ylabel('Count')
plt.xlabel('Review Status')
plt.savefig('review_status_distribution.png')
plt.close()

# Project type distribution
print("\nProject Type Distribution:")
print(df['project_type_id'].value_counts())

# Check for missing values
print("\nMissing Values by Column:")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

In [ ]:
# 3. Text Preprocessing
# ---------------------

def preprocess_text(text, remove_stopwords=True, stem=False, lemmatize=True):
    """
    Preprocess text data: lowercase, remove punctuation, stopwords, and apply stemming/lemmatization
    
    Args:
        text (str): The text to preprocess
        remove_stopwords (bool): Whether to remove stopwords
        stem (bool): Whether to apply stemming
        lemmatize (bool): Whether to apply lemmatization
    
    Returns:
        str: Preprocessed text
    """
    if pd.isna(text) or text == '':
        return ''
    
    # Convert to string if not already
    if not isinstance(text, str):
        text = str(text)
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        tokens = [word for word in tokens if word not in stop_words]
    
    # Apply stemming
    if stem:
        stemmer = PorterStemmer()
        tokens = [stemmer.stem(word) for word in tokens]
    
    # Apply lemmatization
    if lemmatize:
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    # Join tokens back into text
    preprocessed_text = ' '.join(tokens)
    
    return preprocessed_text

# Initialize sentiment analyzer
sia = SentimentIntensityAnalyzer()

# Apply text preprocessing to text fields
print("Applying text preprocessing...")

# Create new columns for preprocessed text
df['title_processed'] = df['title'].apply(preprocess_text)
df['description_processed'] = df['description'].apply(preprocess_text)
df['synopsis_processed'] = df['synopsis'].apply(preprocess_text)

# Generate text features
def extract_text_features(row):
    """Extract features from text fields"""
    # Text length features
    title_length = len(str(row['title'])) if pd.notna(row['title']) else 0
    desc_length = len(str(row['description'])) if pd.notna(row['description']) else 0
    synopsis_length = len(str(row['synopsis'])) if pd.notna(row['synopsis']) else 0
    
    # Word count features
    title_word_count = len(str(row['title']).split()) if pd.notna(row['title']) else 0
    desc_word_count = len(str(row['description']).split()) if pd.notna(row['description']) else 0
    synopsis_word_count = len(str(row['synopsis']).split()) if pd.notna(row['synopsis']) else 0
    
    # Sentiment features (using VADER)
    title_sentiment = sia.polarity_scores(str(row['title']))['compound'] if pd.notna(row['title']) else 0
    desc_sentiment = sia.polarity_scores(str(row['description']))['compound'] if pd.notna(row['description']) else 0
    synopsis_sentiment = sia.polarity_scores(str(row['synopsis']))['compound'] if pd.notna(row['synopsis']) else 0
    
    # Readability features (using textstat)
    desc_readability = textstat.flesch_reading_ease(str(row['description'])) if pd.notna(row['description']) else 0
    synopsis_readability = textstat.flesch_reading_ease(str(row['synopsis'])) if pd.notna(row['synopsis']) else 0
    
    return pd.Series({
        'title_length': title_length,
        'description_length': desc_length,
        'synopsis_length': synopsis_length,
        'title_word_count': title_word_count,
        'description_word_count': desc_word_count,
        'synopsis_word_count': synopsis_word_count,
        'title_sentiment': title_sentiment,
        'description_sentiment': desc_sentiment,
        'synopsis_sentiment': synopsis_sentiment,
        'description_readability': desc_readability,
        'synopsis_readability': synopsis_readability
    })

# Extract text features
print("Extracting text features...")
text_features = df.apply(extract_text_features, axis=1)
df = pd.concat([df, text_features], axis=1)

In [ ]:
import nltk
print(nltk.__version__)
